# L4: Automate Event Planning

In this lesson, you will learn more about Tasks.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Crew, Task

In [3]:
import os
from pathlib import Path
from utils import get_openai_api_key,get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

# Change to workspace root directory first
workspace_root = '/Users/sayantanchakraborty/workspace/poc-workspace/gen-ai-playground'
os.chdir(workspace_root)

# Ensure output directory exists
output_dir = Path('data/generated')
output_dir.mkdir(parents=True, exist_ok=True)

# Use relative paths from workspace root
venue_details_path = 'data/generated/venue_details.json'
marketing_report_path = 'data/generated/marketing_report.md'

print(f"Current working directory: {os.getcwd()}")
print(f"Files will be created at: {Path(venue_details_path).absolute()}")

Current working directory: /Users/sayantanchakraborty/workspace/poc-workspace/gen-ai-playground
Files will be created at: /Users/sayantanchakraborty/workspace/poc-workspace/gen-ai-playground/data/generated/venue_details.json


## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [6]:
 # Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipment"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [7]:
# Agent 3: Marketing and Communications Agent
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

## Creating Venue Pydantic Object

- Create a class `VenueDetails` using [Pydantic BaseModel](https://docs.pydantic.dev/latest/api/base_model/).
- Agents will populate this object with information about different venues by creating different instances of it.

In [8]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

In [9]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file=venue_details_path,  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

- By setting `async_execution=True`, it means the task can run in parallel with the tasks which come after it.

In [10]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    agent=logistics_manager
)

In [11]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file=marketing_report_path,  # Outputs the report as a text file
    agent=marketing_communications_agent
)

## Creating the Crew

**Note**: Since you set `async_execution=True` for `logistics_task` and `marketing_task` tasks, now the order for them does not matter in the `tasks` list.

In [12]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [13]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

**Note 1**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Note 2**: 
- Since you set `human_input=True` for some tasks, the execution will ask for your input before it finishes running.
- When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [14]:
result = event_management_crew.kickoff(inputs=event_details)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2bc3a7c0-6198-4d5e-a096-c93a3faf1973                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Task: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Thought: I need to find a suitable venue in San Francisco that fits the criteria for the Tech Innovation       │
│  Conference. I will search for venues that are tailored for tech conferences in the area.                       │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "Tech Innovation Conference venue San Francisco"                                             │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

 ## Final Result: **Catering Options:**
1. **San Francisco Catering Company**  
   - Website: [sfcateringcompany.com](https://sfcateringcompany.com/)  
   - Description: Offers corporate and event catering with full-service and drop-off options.

2. **Top Caterers in San Francisco**  
   - Website: [PartySlate](https://www.partyslate.com/find-vendors/event-caterer/area/san-francisco)  
   - Description: Features various caterers like On The Roll Catering and Events, McCalls Catering, and Foxtail Catering & Events.

3. **Above & Beyond Catering**  
   - Website: [abovecatering.com](https://abovecatering.com/)  
   - Description: Specializes in weddings and corporate events.

4. **Fraiche Catering**  
   - Website: [fraichecater.com](https://fraichecater.com/)  
   - Description: An award-winning gourmet catering service in the Bay Area.

5. **United Dumplings**  
   - Website: [uniteddumplings.com](https://www.uniteddumplings.com/pages/dumpling-catering-san-francisco-bay)  
   - Descrip

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Logistics Manager                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Catering Options:**                                                                                          │
│  1. **San Francisco Catering Company**                                                                          │
│     - Website: [sfcateringcompany.com](https://sfcateringcompany.com/)                                          │
│     - Description: Offers corporate and event catering with full-service and drop-off options.                  │
│                                                                                                                 │
│  2. **Top Caterers in San Francisco**                                                                           │
│     - Website: [PartySlate](https://www.partyslate.com/find-vendors/event-caterer/area/san-francisco)           │
│     - Description: Features various caterers like On The Roll Catering and Events, McCalls Catering, and        │
│  Foxtail Catering & Events.                                                                                     │
│                                                                                                                 │
│  3. **Above & Beyond Catering**                                                                                 │
│     - Website: [abovecatering.com](https://abovecatering.com/)                                                  │
│     - Description: Specializes in weddings and corporate events.                                                │
│                                                                                                                 │
│  4. **Fraiche Catering**                                                                                        │
│     - Website: [fraichecater.com](https://fraichecater.com/)                                                    │
│     - Description: An award-winning gourmet catering service in the Bay Area.                                   │
│                                                                                                                 │
│  5. **United Dumplings**                                                                                        │
│     - Website:                                                                                                  │
│  [uniteddumplings.com](https://www.uniteddumplings.com/pages/dumpling-catering-san-francisco-bay)               │
│     - Description: Provides full-service catering including menu planning and setup.                            │
│                                                                                                                 │
│  **Equipment Rental Options:**                                                                                  │
│  1. **Abbey Party Rents**                                                                                       │
│     - Website: [abbeyrentssf.com](https://www.abbeyrentssf.com/)                                                │
│     - Description: Offers tents, tables, chairs, and event production equipment.                                │
│                                                                                                                 │
│  2. **Ideas Events & Rentals**                                                                                  │
│     - Website: [ideas-events.com](https://www.ideas-eve

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6b439a84-812f-4e8d-87d9-b2523db272dc                                                                     │
│  Agent: Logistics Manager                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing and Communications Agent                                                                      │
│                                                                                                                 │
│  Thought: I need to gather information about the Tech Innovation Conference to formulate a compelling           │
│  marketing strategy and understand how to engage potential attendees effectively.                               │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2bc3a7c0-6198-4d5e-a096-c93a3faf1973                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # Tech Innovation Conference 2024 Marketing Report                                               │
│                                                                                                                 │
│  ## Event Overview                                                                                              │
│  - **Event Name:** Tech Innovation Conference                                                                   │
│  - **Date:** October 23, 2024                                                                                   │
│  - **Time:** 6:03 PM - 8:03 PM                                                                                  │
│  - **Location:** Moscone Center, Moscone West, 747 Howard St, San Francisco, CA 94103, USA                      │
│  - **Description:** Join tech enthusiasts and industry experts to discover innovation trends shaping the        │
│  future.                                                                                                        │
│                                                                                                                 │
│  ## Marketing Activities                                                                                        │
│  1. **Social Media Campaign:**                                                                                  │
│     - Create engaging posts highlighting speakers, topics, and the venue.                                       │
│     - Use targeted ads to reach tech professionals and enthusiasts.                                             │
│     - Utilize hashtags such as #TechInnovation2024 and #FutureTech to increase visibility.                      │
│                                                                                                                 │
│  2. **Email Marketing:**                                                                                        │
│     - Send out timely newsletters to past attendees, industry partners, and mailing lists.                      │
│     - Include exclusive early bird registration options

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the generated `venue_details.json` file.

In [15]:
import json
from pprint import pprint

with open(venue_details_path) as f:
   data = json.load(f)

pprint(data)

{'address': '900 Marin St, San Francisco, CA 94124',
 'booking_status': 'Available for booking',
 'capacity': 500,
 'name': 'The Midway'}


- Display the generated `marketing_report.md` file.

**Note**: After `kickoff` execution has successfully ran, wait an extra 45 seconds for the `marketing_report.md` file to be generated. If you try to run the code below before the file has been generated, your output would look like:

```
marketing_report.md
```

If you see this output, wait some more and than try again.

In [16]:
from IPython.display import Markdown
Markdown(marketing_report_path)

# Tech Innovation Conference 2024 Marketing Report

## Event Overview
- **Event Name:** Tech Innovation Conference
- **Date:** October 23, 2024
- **Time:** 6:03 PM - 8:03 PM
- **Location:** Moscone Center, Moscone West, 747 Howard St, San Francisco, CA 94103, USA
- **Description:** Join tech enthusiasts and industry experts to discover innovation trends shaping the future. 

## Marketing Activities
1. **Social Media Campaign:**
   - Create engaging posts highlighting speakers, topics, and the venue.
   - Use targeted ads to reach tech professionals and enthusiasts.
   - Utilize hashtags such as #TechInnovation2024 and #FutureTech to increase visibility.

2. **Email Marketing:**
   - Send out timely newsletters to past attendees, industry partners, and mailing lists.
   - Include exclusive early bird registration options and speaker announcements to entice registrations.

3. **Partnerships and Collaboration:**
   - Partner with tech blogs and influencers for promotions.
   - Collaborate with universities and technical institutions to reach students and faculty.

4. **Content Marketing:**
   - Publish articles and blog posts discussing emerging technologies and what to expect at the conference.
   - Share videos or testimonials from past attendees to promote the event's impact.

5. **Event Listing Websites:**
   - List the conference on major event and conference websites for wider exposure.

## Attendee Engagement Strategies
1. **Interactive Sessions:**
   - Include Q&A formats and panel discussions to encourage audience participation.
   - Offer hands-on workshops to provide a deeper dive into specific technologies.

2. **Networking Opportunities:**
   - Facilitate networking sessions before and after the main conference to allow attendees to connect.
   - Use event apps for attendees to interact and schedule meetings.

3. **Feedback Mechanism:**
   - Collect feedback post-event to improve future conferences.
   - Engage with attendees through surveys to understand their interests and expectations better.

4. **Swag Bags:**
   - Distribute swag bags containing tech-related items, brochures, and promotional materials from sponsors.

5. **Follow-up Communication:**
   - Send thank-you emails post-event summarizing highlights and learnings.
   - Invite attendees to join future events and stay connected through newsletters.

By implementing these marketing activities and engagement strategies, we aim to attract at least 500 potential attendees to the Tech Innovation Conference and create a lasting impact within the tech community.